# Vizualise cell assemblies reactivation

Load packages

In [ ]:
from scipy import signal
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.widgets import Slider, Button, Cursor
from scipy import fftpack
import pandas as pd
from pathlib import Path
import os
import copy
import json
from IPython.display import display
from ipyfilechooser import FileChooser
from ephyviewer import mkQApp, MainViewer, TraceViewer
from ephyviewer import AnalogSignalSourceWithScatter
import ephyviewer
from scipy.stats import zscore
from scipy.interpolate import interp1d
from itertools import groupby
import sys 
import pickle
import seaborn as sns
from scipy.signal import find_peaks
from scipy.signal import chirp, find_peaks, peak_widths
from scipy import stats
import matplotlib.colors as mcolors
import warnings
from scipy import interpolate
import re
from scipy.stats import f_oneway
from scipy.spatial.distance import pdist, squareform
from scipy.stats import mannwhitneyu
from scipy.stats import ks_2samp
import diptest
import random
from collections import Counter, defaultdict
import seaborn as sns
import pandas as pd
from matplotlib.colors import ListedColormap
import networkx as nx
from scipy.stats import wilcoxon
from statsmodels.stats.multicomp import pairwise_tukeyhsd
import ast
warnings.filterwarnings("ignore")

%matplotlib widget

## Cell assemblies activity across vigilance states

Plot Cell assembly activity relative to Vig States

In [ ]:
Analysisfolder= '3_Reactivation_2025-11-05_10_23_42_calcium_nozscore' 

In [ ]:
dfilepath=f"//10.69.168.1/crnldata/forgetting/Aurelie/MiniscopeOE_analysis/PlaceCells_experiment/{Analysisfolder}/CellAssembly_Global.pkl"
with open(dfilepath, 'rb') as pickle_file:
    df_CellAss_origin = pickle.load(pickle_file)

drug= 'baseline'
NrSubtype='L2_3_mice' # L1NDNF_mice OR L2_3_mice

In [ ]:
df_CellAss=df_CellAss_origin.copy()
df_CellAss = df_CellAss[df_CellAss['SubstateDuration'] != 0]


initial_nb= df_CellAss['Assembly_ID'].nunique()
df_CellAss = df_CellAss[df_CellAss['Assembly_size'] > 1]
print(initial_nb - df_CellAss['Assembly_ID'].nunique(),'/',initial_nb, 'cell assemblies contained no neurons')

# Convert string representations of lists into actual Python lists.
df_CellAss['Cells_in_Assembly']=df_CellAss['Cells_in_Assembly'] = df_CellAss['Cells_in_Assembly'].apply(lambda x: ast.literal_eval(x) if isinstance(x, str) else x)
# Filter the DataFrame to only keep rows where the number of cells in the list matches the 'Assembly_size'.
df_CellAss = df_CellAss[df_CellAss['Cells_in_Assembly'].apply(len) == df_CellAss['Assembly_size']]

plt.figure(figsize=(4, 3))
table_CellAss = df_CellAss.pivot_table(index='Assembly_ID', columns=[df_CellAss['ExpeType']], values='Avg_ReactStr', aggfunc='mean', fill_value=None)
desired_order = ['Cheeseboard','SleepAfter']   
try: table_CellAss = table_CellAss[desired_order]
except: pass
plt.subplot(1,2,1)
plt.plot(table_CellAss.columns, table_CellAss.values.T, color= 'grey', alpha=0.15, linewidth=2)
plt.plot(table_CellAss.columns, table_CellAss.mean(), linewidth=2, color='black')
plt.errorbar(table_CellAss.columns, table_CellAss.mean(), yerr=table_CellAss.sem() ,
             fmt='o', color='black', capsize=5)
plt.ylabel('Reactivation strength per cell assemblies')

stats.shapiro(table_CellAss['Cheeseboard'])
stats.shapiro(table_CellAss['SleepAfter'])
res= wilcoxon(table_CellAss['Cheeseboard'], table_CellAss['SleepAfter'])
print(res)

table_CellAss2 = df_CellAss.pivot_table(index='Assembly_ID', columns=[df_CellAss['ExpeType']], values='EventFreq', aggfunc='mean', fill_value=None)
try: table_CellAss2 = table_CellAss2[desired_order]
except: pass
plt.subplot(1,2,2)
plt.plot(table_CellAss2.columns, table_CellAss2.values.T, color= 'grey', alpha=0.15, linewidth=2)
plt.plot(table_CellAss2.columns, table_CellAss2.mean(), linewidth=2, color='black')
plt.errorbar(table_CellAss2.columns, table_CellAss2.mean(), yerr=table_CellAss2.sem() ,
            fmt='o', color='black', capsize=5)
plt.ylabel('Event frequency per cell assemblies')
plt.suptitle(f'{NrSubtype}, n={len(table_CellAss)}, All VigSt')
plt.tight_layout()
plt.savefig(f'{Path(dfilepath).parent}/CellAssembly_MazeVSPost.svg', format='svg', bbox_inches='tight', pad_inches=0, transparent=True)
plt.show()

stats.shapiro(table_CellAss2['Cheeseboard'])
stats.shapiro(table_CellAss2['SleepAfter'])
res= wilcoxon(table_CellAss2['Cheeseboard'], table_CellAss2['SleepAfter'])
print(res)

In [ ]:
df_CellAss=df_CellAss_origin.copy()
df_CellAss = df_CellAss[df_CellAss['SubstateDuration'] != 0]

df_CellAss = df_CellAss[df_CellAss['Substate']=='AW']

initial_nb= df_CellAss['Assembly_ID'].nunique()
df_CellAss = df_CellAss[df_CellAss['Assembly_size'] > 1]
print(initial_nb - df_CellAss['Assembly_ID'].nunique(),'/',initial_nb, 'cell assemblies contained no neurons')

# Convert string representations of lists into actual Python lists.
df_CellAss['Cells_in_Assembly']=df_CellAss['Cells_in_Assembly'] = df_CellAss['Cells_in_Assembly'].apply(lambda x: ast.literal_eval(x) if isinstance(x, str) else x)
# Filter the DataFrame to only keep rows where the number of cells in the list matches the 'Assembly_size'.
df_CellAss = df_CellAss[df_CellAss['Cells_in_Assembly'].apply(len) == df_CellAss['Assembly_size']]

plt.figure(figsize=(4, 3))
table_CellAss = df_CellAss.pivot_table(index='Assembly_ID', columns=[df_CellAss['ExpeType']], values='Avg_ReactStr', aggfunc='mean', fill_value=None)
desired_order = ['Cheeseboard','SleepAfter']   
try: table_CellAss = table_CellAss[desired_order]
except: pass
plt.subplot(1,2,1)
plt.plot(table_CellAss.columns, table_CellAss.values.T, color= 'grey', alpha=0.15, linewidth=2)
plt.plot(table_CellAss.columns, table_CellAss.mean(), linewidth=2, color='black')
plt.errorbar(table_CellAss.columns, table_CellAss.mean(), yerr=table_CellAss.sem() ,
             fmt='o', color='black', capsize=5)
plt.ylabel('Reactivation strength per cell assemblies')

stats.shapiro(table_CellAss['Cheeseboard'])
stats.shapiro(table_CellAss['SleepAfter'])
res= wilcoxon(table_CellAss['Cheeseboard'], table_CellAss['SleepAfter'])
print(res)

table_CellAss2 = df_CellAss.pivot_table(index='Assembly_ID', columns=[df_CellAss['ExpeType']], values='EventFreq', aggfunc='mean', fill_value=None)
try: table_CellAss2 = table_CellAss2[desired_order]
except: pass
plt.subplot(1,2,2)
plt.plot(table_CellAss2.columns, table_CellAss2.values.T, color= 'grey', alpha=0.15, linewidth=2)
plt.plot(table_CellAss2.columns, table_CellAss2.mean(), linewidth=2, color='black')
plt.errorbar(table_CellAss2.columns, table_CellAss2.mean(), yerr=table_CellAss2.sem() ,
            fmt='o', color='black', capsize=5)
plt.ylabel('Event frequency per cell assemblies')
plt.suptitle(f'{NrSubtype}, n={len(table_CellAss)}, AW')
plt.tight_layout()
plt.savefig(f'{Path(dfilepath).parent}/CellAssembly_AWMazeVSAWPost.svg', format='svg', bbox_inches='tight', pad_inches=0, transparent=True)
plt.show()

stats.shapiro(table_CellAss2['Cheeseboard'])
stats.shapiro(table_CellAss2['SleepAfter'])
res= wilcoxon(table_CellAss2['Cheeseboard'], table_CellAss2['SleepAfter'])
print(res)

In [ ]:
df_CellAss=df_CellAss_origin.copy()
df_CellAss = df_CellAss[df_CellAss['SubstateDuration'] != 0]

df_CellAss = df_CellAss[df_CellAss['ExpeType']=='SleepAfter']

initial_nb= df_CellAss['Assembly_ID'].nunique()
df_CellAss = df_CellAss[df_CellAss['Assembly_size'] > 1]
print(initial_nb - df_CellAss['Assembly_ID'].nunique(),'/',initial_nb, 'cell assemblies contained no neurons')

# Convert string representations of lists into actual Python lists.
df_CellAss['Cells_in_Assembly']=df_CellAss['Cells_in_Assembly'] = df_CellAss['Cells_in_Assembly'].apply(lambda x: ast.literal_eval(x) if isinstance(x, str) else x)
# Filter the DataFrame to only keep rows where the number of cells in the list matches the 'Assembly_size'.
df_CellAss = df_CellAss[df_CellAss['Cells_in_Assembly'].apply(len) == df_CellAss['Assembly_size']]

plt.figure(figsize=(4, 3))
table_CellAss = df_CellAss.pivot_table(index='Assembly_ID', columns=[df_CellAss['Substate']], values='Avg_ReactStr', aggfunc='mean', fill_value=None)
desired_order = ['AW','QW', 'NREM', 'REM']   
try:del table_CellAss['undefined']
except: pass
try:del table_CellAss['IS']
except: pass
try: table_CellAss = table_CellAss[desired_order]
except: pass
plt.subplot(1,2,1)
plt.plot(table_CellAss.columns, table_CellAss.values.T, color= 'grey', alpha=0.15, linewidth=2)
plt.plot(table_CellAss.columns, table_CellAss.mean(), linewidth=2, color='black')
plt.errorbar(table_CellAss.columns, table_CellAss.mean(), yerr=table_CellAss.sem() ,
             fmt='o', color='black', capsize=5)
plt.ylabel('Reactivation strength per cell assemblies')
plt.tight_layout()

groups = [table_CellAss[col].dropna().values for col in table_CellAss.columns]
f_stat, p_value = f_oneway(*groups)
print(f"ANOVA result: F = {f_stat:.3f}, p = {p_value:.3e}")
df_melt = table_CellAss.melt(var_name='Substates', value_name='Avg_Activity')
df_melt = df_melt.dropna(subset=['Avg_Activity', 'Substates'])
df_melt['Avg_Activity'] = pd.to_numeric(df_melt['Avg_Activity'], errors='coerce')
tukey = pairwise_tukeyhsd(endog=df_melt['Avg_Activity'], groups=df_melt['Substates'], alpha=0.05)
print(tukey.summary())

table_CellAss = df_CellAss.pivot_table(index='Assembly_ID', columns=[df_CellAss['Substate']], values='EventFreq', aggfunc='mean', fill_value=None)
try:del table_CellAss['undefined']
except: pass
try:del table_CellAss['IS']
except: pass
try: table_CellAss = table_CellAss[desired_order]
except: pass
plt.subplot(1,2,2)
plt.plot(table_CellAss.columns, table_CellAss.values.T, color= 'grey', alpha=0.15, linewidth=2)
plt.plot(table_CellAss.columns, table_CellAss.mean(), linewidth=2, color='black')
plt.errorbar(table_CellAss.columns, table_CellAss.mean(), yerr=table_CellAss.sem() ,
            fmt='o', color='black', capsize=5)
plt.ylabel('Event frequency per cell assemblies')

groups = [table_CellAss[col].dropna().values for col in table_CellAss.columns]
f_stat, p_value = f_oneway(*groups)
print(f"ANOVA result: F = {f_stat:.3f}, p = {p_value:.3e}")
df_melt = table_CellAss.melt(var_name='Substates', value_name='EventFreq')
df_melt = df_melt.dropna(subset=['EventFreq', 'Substates'])
df_melt['EventFreq'] = pd.to_numeric(df_melt['EventFreq'], errors='coerce')
tukey = pairwise_tukeyhsd(endog=df_melt['EventFreq'], groups=df_melt['Substates'], alpha=0.05)
print(tukey.summary())
plt.suptitle(f'{NrSubtype}, n={len(table_CellAss)}, SleepAfter')
plt.tight_layout()
plt.savefig(f'{Path(dfilepath).parent}/CellAssembly_VigStActivity.svg', format='svg', bbox_inches='tight', pad_inches=0, transparent=True)
plt.show()

Load NeuronIdentity file & assign Neuron ID to each neuron

In [ ]:
AnalysisfolderVig= '1_VigSt_2025-10-25_16_35_21'
AnalysisfolderNeuronID= '0_NeuronIdentity_2025-10-10_11_13_40'

In [ ]:
with open(f"//10.69.168.1/crnldata/forgetting/Aurelie/MiniscopeOE_analysis/PlaceCells_experiment/{AnalysisfolderNeuronID}/List_SignCells.pkl", 'rb') as pickle_file:
    df_MazeCellID = pickle.load(pickle_file)

AllCheeseboard_cells=np.array(df_MazeCellID['all'])
HD_cells=np.array(df_MazeCellID['HD'])
PC_cells=np.array(df_MazeCellID['PC'])
PCHD_cells=np.array(df_MazeCellID['PC&HD'])

Spatial_cells=np.concatenate([PCHD_cells,  HD_cells, PC_cells])

dfilepathV=f"//10.69.168.1/crnldata/forgetting/Aurelie/MiniscopeOE_analysis/PlaceCells_experiment/{AnalysisfolderVig}/VigStates_Global.pkl"
with open(dfilepathV, 'rb') as pickle_file:
    df_VigSt = pickle.load(pickle_file)

All_cells= np.array(df_VigSt['Unit_ID'].unique().tolist())

id_to_cluster = {
    p: (
        'HD' if p in HD_cells.tolist()
        else 'PC' if p in PC_cells.tolist()
        else 'PC&HD' if p in PCHD_cells.tolist()
        else 'notPCnotHD' if p in AllCheeseboard_cells.tolist() and p not in (HD_cells.tolist() + PC_cells.tolist() + PCHD_cells.tolist())
        else 'unclassified'
    )
    for p in All_cells.tolist()
}
notPCnotHD_cells = np.array([name for name, classification in id_to_cluster.items() if classification == 'notPCnotHD'])
unclassified_cells = np.array([name for name, classification in id_to_cluster.items() if classification == 'unclassified'])


Spatial_id_to_cluster = {
    p: (
        'Spatial' if p in Spatial_cells.tolist()
        else 'Non_Spatial' if p in AllCheeseboard_cells.tolist() and p not in (HD_cells.tolist() + PC_cells.tolist() + PCHD_cells.tolist())
        else 'unclassified'
    )
    for p in All_cells.tolist()
}

Non_Spatial_cells = np.array([name for name, classification in Spatial_id_to_cluster.items() if classification == 'Non_Spatial'])

Save Vigilance State with Neuron ID

In [ ]:
df_VigSt['Neuron_Identity'] = df_VigSt['Unit_ID'].map(id_to_cluster)
df_VigSt['Spatially_tuned'] = df_VigSt['Unit_ID'].map(Spatial_id_to_cluster)

filenameOut = Path(dfilepathV).parent / f'VigStates_Global_NeuronID.pkl'
with open(filenameOut, 'wb') as pickle_file:
    pickle.dump(df_VigSt, pickle_file)
    
filenameOut = Path(dfilepathV).parent / f'VigStates_Global_NeuronID.xlsx'
with pd.ExcelWriter(filenameOut) as writer:
    df_VigSt.to_excel(writer, index=False)

Identify Cell assembly Types and test proportion

In [ ]:
df_CellAss_uniqAss = df_CellAss.drop_duplicates(subset='Assembly_ID', keep='first')
df_CellAss_uniqAss = df_CellAss_uniqAss[df_CellAss_uniqAss['Cells_in_Assembly'].apply(len) == df_CellAss_uniqAss['Assembly_size']]
groups = df_CellAss_uniqAss['Cells_in_Assembly'].tolist()
ids=AllCheeseboard_cells.tolist()
cluster_labels = ['HD']*len(HD_cells) + ['PC']*len(PC_cells) + ['PC&HD']*len(PCHD_cells) + ['notPCnotHD']*len(notPCnotHD_cells) 

def classify_groups(groups, id_to_cluster):
    labels = []
    for group in groups:
        clusters = [id_to_cluster.get(i, np.nan) for i in group]
        if len(clusters) >= 1:
            count = Counter(clusters)
            top, n_top = count.most_common(1)[0]
            top_clusters = [k for k, v in count.items() if v == n_top]        
            # If there are multiple top clusters or the majority is less than 50%, label as "mix"
            if len(top_clusters) == 1 and (n_top / len(group) * 100) > 50 and not pd.isna(top):
                labels.append(top)
            else:
                labels.append("mix")  
        else:
            labels.append("mix")    
    return labels

# --- Observed classification
observed_labels = classify_groups(groups, id_to_cluster)
observed_counts = Counter(observed_labels)
df_CellAss_uniqAss['Assembly_Neurons_ID']= observed_labels

# --- Permutation test
n_perms = 5000
null_dists = defaultdict(list)

for _ in range(n_perms):
    # Shuffle cluster labels across individuals
    shuffled_clusters = dict(zip(ids, random.sample(cluster_labels, len(cluster_labels))))
    shuffled_labels = classify_groups(groups, shuffled_clusters)
    counts = Counter(shuffled_labels)
    for label in observed_counts:
        null_dists[label].append(counts.get(label, 0))

# --- Z-scores and p-values
results = []
for label in observed_counts:
    obs = observed_counts[label]
    null = null_dists[label]
    mean = np.mean(null)
    std = np.std(null)
    z = (obs - mean) / std if std > 0 else 0
    # Two-tailed p-value
    p = (np.sum(np.abs(np.array(null) - mean) >= abs(obs - mean)) + 1) / (n_perms + 1)
    results.append((label, obs, mean, std, z, p))

# --- Output
print("\nAssembly Neuron ID Distribution Test")
print(f"{'ID':>10} {'Obs':>5} {'Mean':>7} {'Std':>7} {'Z':>6} {'p-value':>8}")
for label, obs, mean, std, z, p in sorted(results, key=lambda x: x[-1]):
    print(f"{label:>10} {obs:5} {mean:7.2f} {std:7.2f} {z:6.2f} {p:8.4f}")


df_perm = pd.DataFrame(results, columns=["Cell_Assembly_ID", "Obs", "PermMean", "Std", "Z", "p"])
df_perm['Obs_prop'] = df_perm['Obs'] / df_perm['Obs'].sum() *100
df_perm['Perm_prop'] = df_perm['PermMean'] / df_perm['PermMean'].sum() *100
df_perm["Perm_std"] = df_perm["Std"] / df_perm["PermMean"].sum() *100

df_perm = df_perm.sort_values("Cell_Assembly_ID") 
df_perm['Cell_Assembly_ID'] = pd.Categorical(df_perm['Cell_Assembly_ID'], ["PC", "HD", "PC&HD", "notPCnotHD", "mix"])
df_perm = df_perm.sort_values("Cell_Assembly_ID") 
df_perm['Cell_Assembly_ID'] = pd.Categorical(df_perm['Cell_Assembly_ID'], ["PC", "HD", "PC&HD", "notPCnotHD", "mix"])
fig, ax= plt.subplots(figsize=(3, 3))
color_map = ['blue',  'green', 'orange','black','purple']
for x, yperm, yobs, err, color in zip(df_perm["Cell_Assembly_ID"], df_perm["Perm_prop"], df_perm["Obs_prop"], df_perm["Perm_std"], color_map):
    line1 = ax.errorbar(x, yperm, yerr=err, fmt='o', color=color, capsize=5, alpha = 0.5, label = 'Permuted ± std')
    line2 = ax.scatter(x, yobs, color=color, label = 'Observed')
    ax.legend([line2, line1], ['Observed', 'Permuted ± std']) if x == 'mix' else None
plt.ylabel("Cell assemblies (%)")
plt.ylim(0, 100)
plt.title(f"Cell Assembly Types (n={len(df_CellAss_uniqAss)})")
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.savefig(f'{Path(dfilepath).parent}/CellAssembly_Proportion_permvsreal_{drug}.svg', format='svg', bbox_inches='tight', pad_inches=0, transparent=True)
plt.show()

In [ ]:
df_SpCellAss_uniqAss = df_CellAss.drop_duplicates(subset='Assembly_ID', keep='first')
df_SpCellAss_uniqAss = df_SpCellAss_uniqAss[df_SpCellAss_uniqAss['Cells_in_Assembly'].apply(len) == df_SpCellAss_uniqAss['Assembly_size']]
groups = df_SpCellAss_uniqAss['Cells_in_Assembly'].tolist()
ids=AllCheeseboard_cells.tolist()
cluster_labels = ['Spatial']*len(Spatial_cells) + ['Non_Spatial']*len(Non_Spatial_cells)

def classify_groups(groups, Spatial_id_to_cluster):
    labels = []
    for group in groups:
        clusters = [Spatial_id_to_cluster.get(i, np.nan) for i in group]
        if len(clusters) >= 1:
            count = Counter(clusters)
            top, n_top = count.most_common(1)[0]
            top_clusters = [k for k, v in count.items() if v == n_top]        
            # If there are multiple top clusters or the majority is less than 50%, label as "mix"
            if len(top_clusters) == 1 and (n_top / len(group) * 100) > 50 and not pd.isna(top):
                labels.append(top)
            else:
                labels.append("mix")  
        else:
            labels.append("mix")    
    return labels

# --- Observed classification
observed_labels = classify_groups(groups, Spatial_id_to_cluster)
observed_counts = Counter(observed_labels)
df_SpCellAss_uniqAss['Assembly_Spatial_ID']= observed_labels

# --- Permutation test
n_perms = 5000
null_dists = defaultdict(list)

for _ in range(n_perms):
    # Shuffle cluster labels across individuals
    shuffled_clusters = dict(zip(ids, random.sample(cluster_labels, len(cluster_labels))))
    shuffled_labels = classify_groups(groups, shuffled_clusters)
    counts = Counter(shuffled_labels)
    for label in observed_counts:
        null_dists[label].append(counts.get(label, 0))

# --- Z-scores and p-values
results = []
for label in observed_counts:
    obs = observed_counts[label]
    null = null_dists[label]
    mean = np.mean(null)
    std = np.std(null)
    z = (obs - mean) / std if std > 0 else 0
    # Two-tailed p-value
    p = (np.sum(np.abs(np.array(null) - mean) >= abs(obs - mean)) + 1) / (n_perms + 1)
    results.append((label, obs, mean, std, z, p))

# --- Output
print("\nAssembly Neuron ID Distribution Test")
print(f"{'ID':>10} {'Obs':>5} {'Mean':>7} {'Std':>7} {'Z':>6} {'p-value':>8}")
for label, obs, mean, std, z, p in sorted(results, key=lambda x: x[-1]):
    print(f"{label:>10} {obs:5} {mean:7.2f} {std:7.2f} {z:6.2f} {p:8.4f}")


df_perm = pd.DataFrame(results, columns=["Cell_Assembly_ID", "Obs", "PermMean", "Std", "Z", "p"])
df_perm['Obs_prop'] = df_perm['Obs'] / df_perm['Obs'].sum() *100
df_perm['Perm_prop'] = df_perm['PermMean'] / df_perm['PermMean'].sum() *100
df_perm["Perm_std"] = df_perm["Std"] / df_perm["PermMean"].sum() *100
df_perm = df_perm.sort_values("Cell_Assembly_ID") 
df_perm['Cell_Assembly_ID'] = pd.Categorical(df_perm['Cell_Assembly_ID'], ["Spatial", "Non_Spatial","mix"])
df_perm = df_perm.sort_values("Cell_Assembly_ID") 
df_perm['Cell_Assembly_ID'] = pd.Categorical(df_perm['Cell_Assembly_ID'], ["Spatial", "Non_Spatial", "mix"])

fig, ax= plt.subplots(figsize=(3, 3))
color_map = ['blue', 'orange', 'purple']
for x, yperm, yobs, err, color in zip(df_perm["Cell_Assembly_ID"], df_perm["Perm_prop"], df_perm["Obs_prop"], df_perm["Perm_std"], color_map):
    line1= ax.errorbar(x, yperm, yerr=err, fmt='o', color=color, capsize=5, alpha = 0.5, label = 'Permuted ± std')
    line2= ax.scatter(x, yobs, color=color, label = 'Observed')
    ax.legend([line2, line1], ['Observed', 'Permuted ± std']) if x == 'mix' else None
plt.ylabel("Cell assemblies (%)")
plt.ylim(0, 100)
plt.title(f"Cell Assembly Types (n={len(df_SpCellAss_uniqAss)})")
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.savefig(f'{Path(dfilepath).parent}/SpatialCellAssembly_Proportion_permvsreal_{drug}.svg', format='svg', bbox_inches='tight', pad_inches=0, transparent=True)
plt.show()

Identify modulated cell assemblies during Post Session

In [ ]:
df_CellAss = df_CellAss_origin.copy()
df_CellAss = df_CellAss[df_CellAss['SubstateDuration'] != 0]

initial_nb= df_CellAss['Assembly_ID'].nunique()
df_CellAss = df_CellAss[df_CellAss['Assembly_size'] > 1]
df_CellAss['Cells_in_Assembly']=df_CellAss['Cells_in_Assembly'] = df_CellAss['Cells_in_Assembly'].apply(lambda x: ast.literal_eval(x) if isinstance(x, str) else x)
df_CellAss = df_CellAss[df_CellAss['Cells_in_Assembly'].apply(len) == df_CellAss['Assembly_size']]


table_CellAss = df_CellAss.pivot_table(index='Assembly_ID', columns=[df_CellAss['ExpeType']], values='Avg_ReactStr', aggfunc='mean', fill_value=None)
table_CellAss['Diff']=(table_CellAss['Cheeseboard']-table_CellAss['SleepAfter'])/table_CellAss['Cheeseboard']*100
table_CellAss['Modulated_in_Post']=np.where(table_CellAss['Diff'] > 20, 'Negatively', np.where(table_CellAss['Diff'] < -20, 'Positively', 'NotModulated'))

plt.close('all')

df_CellAss_uniqAss_Mod = pd.merge(df_CellAss_uniqAss, table_CellAss[['Modulated_in_Post']], on='Assembly_ID', how='outer')

fig, axes = plt.subplots(1, 3, figsize=(7, 3.5))
table_ = df_CellAss.pivot_table(index='Assembly_ID', columns=[df_CellAss['ExpeType']], values='Avg_ReactStr', aggfunc='mean', fill_value=None)
table_ = pd.merge(table_,df_CellAss_uniqAss_Mod[['Assembly_ID', 'Assembly_Neurons_ID']], on='Assembly_ID', how='outer')

desired_order = ['Cheeseboard','SleepAfter', 'Assembly_Neurons_ID']   
try: table_ = table_[desired_order]
except: pass
color_map = {'PC': 'blue', 'PC&HD': 'orange', 'HD': 'green', 'notPCnotHD': 'grey', 'mix': 'purple'}
for idx, row in table_.iterrows():
    axes[0].plot(table_.columns[:-1], [row['Cheeseboard'], row['SleepAfter']], color=color_map[row['Assembly_Neurons_ID']], alpha=0.5, linewidth=2)
axes[0].plot(table_.columns[:-1], table_.iloc[:, :-1].mean(), linewidth=2, color='black')
axes[0].errorbar(table_.columns[:-1], table_.iloc[:, :-1].mean(), yerr=table_.iloc[:, :-1].sem(), fmt='o', color='black', capsize=5)
axes[0].set_ylabel('Reactivation strength')
axes[0].set_xticklabels(table_.columns[:-1], rotation=45, ha='right')

color_m = ['blue', 'orange', 'green', 'grey', 'purple']
grouped_proportions = df_CellAss_uniqAss_Mod.groupby('Modulated_in_Post')['Assembly_Neurons_ID'].value_counts(normalize=False).unstack()
desired_order = ['PC', 'PC&HD', 'HD', 'notPCnotHD', 'mix']   
try: grouped_proportions = grouped_proportions[desired_order]
except: pass
grouped_proportions.plot(kind='bar', stacked=True, ax=axes[1], color=color_m)
axes[1].set_ylabel('Count')
axes[1].get_legend().remove()  
axes[1].set_xticklabels(grouped_proportions.index, rotation=45, ha='right')

grouped_proportions = df_CellAss_uniqAss_Mod.groupby('Modulated_in_Post')['Assembly_Neurons_ID'].value_counts(normalize=True).unstack()*100
desired_order = ['PC', 'PC&HD', 'HD', 'notPCnotHD', 'mix']   
try: grouped_proportions = grouped_proportions[desired_order]
except: pass
grouped_proportions.plot(kind='bar', stacked=True, ax=axes[2], color=color_m)
axes[2].set_ylabel('Proportion (%)')
axes[2].legend(loc='center left', bbox_to_anchor=(1, 0.5))
axes[2].set_xticklabels(grouped_proportions.index, rotation=45, ha='right')

fig.suptitle(f'{NrSubtype}, n={len(table_)}, All VigSt')
plt.tight_layout()
plt.savefig(f'{Path(dfilepath).parent}/CellAssembly_Modulated_PostAllVigSt.svg', format='svg', bbox_inches='tight', pad_inches=0, transparent=True)
plt.show()



df_SpCellAss_uniqAss_Mod = pd.merge(df_SpCellAss_uniqAss,table_CellAss[['Modulated_in_Post']], on='Assembly_ID', how='outer')

fig, axes = plt.subplots(1, 3, figsize=(7, 3.5))
table_ = df_CellAss.pivot_table(index='Assembly_ID', columns=[df_CellAss['ExpeType']], values='Avg_ReactStr', aggfunc='mean', fill_value=None)
table_ = pd.merge(table_,df_SpCellAss_uniqAss_Mod[['Assembly_ID', 'Assembly_Spatial_ID']], on='Assembly_ID', how='outer')

desired_order = ['Cheeseboard','SleepAfter', 'Assembly_Spatial_ID']   
try: table_ = table_[desired_order]
except: pass
color_map = {'Spatial': 'blue', 'Non_Spatial': 'orange', 'mix': 'purple'}
for idx, row in table_.iterrows():
    axes[0].plot(table_.columns[:-1], [row['Cheeseboard'], row['SleepAfter']], color=color_map[row['Assembly_Spatial_ID']], alpha=0.5, linewidth=2)
axes[0].plot(table_.columns[:-1], table_.iloc[:, :-1].mean(), linewidth=2, color='black')
axes[0].errorbar(table_.columns[:-1], table_.iloc[:, :-1].mean(), yerr=table_.iloc[:, :-1].sem(), fmt='o', color='black', capsize=5)
axes[0].set_ylabel('Reactivation strength')
axes[0].set_xticklabels(table_.columns[:-1], rotation=45, ha='right')

grouped_proportions = df_SpCellAss_uniqAss_Mod.groupby('Modulated_in_Post')['Assembly_Spatial_ID'].value_counts(normalize=False).unstack()
color_m = ['blue', 'orange', 'purple']
desired_order = ['Spatial', 'Non_Spatial', 'mix']   
try: grouped_proportions = grouped_proportions[desired_order]
except: pass
grouped_proportions.plot(kind='bar', stacked=True, ax=axes[1], color=color_m)
axes[1].set_ylabel('Count')
axes[1].get_legend().remove()  
axes[1].set_xticklabels(grouped_proportions.index, rotation=45, ha='right')

grouped_proportions = df_SpCellAss_uniqAss_Mod.groupby('Modulated_in_Post')['Assembly_Spatial_ID'].value_counts(normalize=True).unstack()*100
desired_order = ['Spatial', 'Non_Spatial', 'mix']   
try: grouped_proportions = grouped_proportions[desired_order]
except: pass
grouped_proportions.plot(kind='bar', stacked=True, ax=axes[2],color=color_m)
axes[2].set_ylabel('Proportion (%)')
axes[2].legend(loc='center left', bbox_to_anchor=(1, 0.5))
axes[2].set_xticklabels(grouped_proportions.index, rotation=45, ha='right')

fig.suptitle(f'{NrSubtype}, n={len(table_)}, All VigSt')
plt.tight_layout()
plt.savefig(f'{Path(dfilepath).parent}/CellAssemblySpatial_Modulated_PostAllVigSt.svg', format='svg', bbox_inches='tight', pad_inches=0, transparent=True)
plt.show()


Identify modulated cell assemblies during AW Post Session

In [ ]:
df_CellAssAW = df_CellAss_origin.copy()
df_CellAssAW = df_CellAssAW[df_CellAssAW['SubstateDuration'] != 0]

df_CellAssAW = df_CellAssAW[df_CellAssAW['Substate']=='AW']

initial_nb= df_CellAssAW['Assembly_ID'].nunique()
df_CellAssAW = df_CellAssAW[df_CellAssAW['Assembly_size'] > 1]
df_CellAssAW['Cells_in_Assembly']=df_CellAssAW['Cells_in_Assembly'] = df_CellAssAW['Cells_in_Assembly'].apply(lambda x: ast.literal_eval(x) if isinstance(x, str) else x)
df_CellAssAW = df_CellAssAW[df_CellAssAW['Cells_in_Assembly'].apply(len) == df_CellAssAW['Assembly_size']]


table_CellAss = df_CellAssAW.pivot_table(index='Assembly_ID', columns=[df_CellAssAW['ExpeType']], values='Avg_ReactStr', aggfunc='mean', fill_value=None)
table_CellAss['Diff']=(table_CellAss['Cheeseboard']-table_CellAss['SleepAfter'])/table_CellAss['Cheeseboard']*100
table_CellAss['Modulated_in_AW']=np.where(table_CellAss['Diff'] > 20, 'Negatively', np.where(table_CellAss['Diff'] < -20, 'Positively', 'NotModulated'))

plt.close('all')

df_CellAss_uniqAss_Mod2 = pd.merge(df_CellAss_uniqAss, table_CellAss[['Modulated_in_AW']], on='Assembly_ID', how='outer')

fig, axes = plt.subplots(1, 3, figsize=(7, 3.5))
table_ = df_CellAssAW.pivot_table(index='Assembly_ID', columns=[df_CellAssAW['ExpeType']], values='Avg_ReactStr', aggfunc='mean', fill_value=None)
table_ = pd.merge(table_,df_CellAss_uniqAss_Mod2[['Assembly_ID', 'Assembly_Neurons_ID']], on='Assembly_ID', how='outer')

desired_order = ['Cheeseboard','SleepAfter', 'Assembly_Neurons_ID']   
try: table_ = table_[desired_order]
except: pass
color_map = {'PC': 'blue', 'PC&HD': 'orange', 'HD': 'green', 'notPCnotHD': 'grey', 'mix': 'purple'}
for idx, row in table_.iterrows():
    axes[0].plot(table_.columns[:-1], [row['Cheeseboard'], row['SleepAfter']], color=color_map[row['Assembly_Neurons_ID']], alpha=0.5, linewidth=2)
axes[0].plot(table_.columns[:-1], table_.iloc[:, :-1].mean(), linewidth=2, color='black')
axes[0].errorbar(table_.columns[:-1], table_.iloc[:, :-1].mean(), yerr=table_.iloc[:, :-1].sem(), fmt='o', color='black', capsize=5)
axes[0].set_ylabel('Reactivation strength')
axes[0].set_xticklabels(table_.columns[:-1], rotation=45, ha='right')

color_m = ['blue', 'orange', 'green', 'grey', 'purple']
grouped_proportions = df_CellAss_uniqAss_Mod2.groupby('Modulated_in_AW')['Assembly_Neurons_ID'].value_counts(normalize=False).unstack()
desired_order = ['PC', 'PC&HD', 'HD', 'notPCnotHD', 'mix']   
try: grouped_proportions = grouped_proportions[desired_order]
except: pass
grouped_proportions.plot(kind='bar', stacked=True, ax=axes[1], color=color_m)
axes[1].set_ylabel('Count')
axes[1].get_legend().remove()  
axes[1].set_xticklabels(grouped_proportions.index, rotation=45, ha='right')

grouped_proportions = df_CellAss_uniqAss_Mod2.groupby('Modulated_in_AW')['Assembly_Neurons_ID'].value_counts(normalize=True).unstack()*100
desired_order = ['PC', 'PC&HD', 'HD', 'notPCnotHD', 'mix']   
try: grouped_proportions = grouped_proportions[desired_order]
except: pass
grouped_proportions.plot(kind='bar', stacked=True, ax=axes[2], color=color_m)
axes[2].set_ylabel('Proportion (%)')
axes[2].legend(loc='center left', bbox_to_anchor=(1, 0.5))
axes[2].set_xticklabels(grouped_proportions.index, rotation=45, ha='right')

fig.suptitle(f'{NrSubtype}, n={len(table_)}, AW')
plt.tight_layout()
plt.savefig(f'{Path(dfilepath).parent}/CellAssembly_Modulated_Post_AW.svg', format='svg', bbox_inches='tight', pad_inches=0, transparent=True)
plt.show()



df_SpCellAss_uniqAss_Mod2 = pd.merge(df_SpCellAss_uniqAss,table_CellAss[['Modulated_in_AW']], on='Assembly_ID', how='outer')

fig, axes = plt.subplots(1, 3, figsize=(7, 3.5))
table_ = df_CellAss.pivot_table(index='Assembly_ID', columns=[df_CellAss['ExpeType']], values='Avg_ReactStr', aggfunc='mean', fill_value=None)
table_ = pd.merge(table_,df_SpCellAss_uniqAss_Mod2[['Assembly_ID', 'Assembly_Spatial_ID']], on='Assembly_ID', how='outer')

desired_order = ['Cheeseboard','SleepAfter', 'Assembly_Spatial_ID']   
try: table_ = table_[desired_order]
except: pass
color_map = {'Spatial': 'blue', 'Non_Spatial': 'orange', 'mix': 'purple'}
for idx, row in table_.iterrows():
    axes[0].plot(table_.columns[:-1], [row['Cheeseboard'], row['SleepAfter']], color=color_map[row['Assembly_Spatial_ID']], alpha=0.5, linewidth=2)
axes[0].plot(table_.columns[:-1], table_.iloc[:, :-1].mean(), linewidth=2, color='black')
axes[0].errorbar(table_.columns[:-1], table_.iloc[:, :-1].mean(), yerr=table_.iloc[:, :-1].sem(), fmt='o', color='black', capsize=5)
axes[0].set_ylabel('Reactivation strength')
axes[0].set_xticklabels(table_.columns[:-1], rotation=45, ha='right')

grouped_proportions = df_SpCellAss_uniqAss_Mod2.groupby('Modulated_in_AW')['Assembly_Spatial_ID'].value_counts(normalize=False).unstack()
color_m = ['blue', 'orange', 'purple']
desired_order = ['Spatial', 'Non_Spatial', 'mix']   
try: grouped_proportions = grouped_proportions[desired_order]
except: pass
grouped_proportions.plot(kind='bar', stacked=True, ax=axes[1], color=color_m)
axes[1].set_ylabel('Count')
axes[1].get_legend().remove()  
axes[1].set_xticklabels(grouped_proportions.index, rotation=45, ha='right')

grouped_proportions = df_SpCellAss_uniqAss_Mod2.groupby('Modulated_in_AW')['Assembly_Spatial_ID'].value_counts(normalize=True).unstack()*100
desired_order = ['Spatial', 'Non_Spatial', 'mix']   
try: grouped_proportions = grouped_proportions[desired_order]
except: pass
grouped_proportions.plot(kind='bar', stacked=True, ax=axes[2],color=color_m)
axes[2].set_ylabel('Proportion (%)')
axes[2].legend(loc='center left', bbox_to_anchor=(1, 0.5))
axes[2].set_xticklabels(grouped_proportions.index, rotation=45, ha='right')

fig.suptitle(f'{NrSubtype}, n={len(table_)}, AW')
plt.tight_layout()
plt.savefig(f'{Path(dfilepath).parent}/CellAssemblySpatial_Modulated_Post_AW.svg', format='svg', bbox_inches='tight', pad_inches=0, transparent=True)
plt.show()


Add & save Assembly main identity

In [ ]:
df_CellAss_ID_ = pd.merge(df_CellAss,df_CellAss_uniqAss_Mod[['Assembly_ID', 'Assembly_Neurons_ID','Modulated_in_Post']], on='Assembly_ID', how='outer')
df_CellAss_ID = pd.merge(df_CellAss_ID_,df_SpCellAss_uniqAss_Mod2[['Assembly_ID', 'Assembly_Spatial_ID', 'Modulated_in_AW']], on='Assembly_ID', how='outer')

filenameOut = Path(dfilepath).parent / f'CellAssembly_Global_ID.pkl'
with open(filenameOut, 'wb') as pickle_file:
    pickle.dump(df_CellAss_ID, pickle_file)    
filenameOut = Path(dfilepath).parent / f'CellAssembly_Global_ID.xlsx'
with pd.ExcelWriter(filenameOut) as writer:
    df_CellAss_ID.to_excel(writer, index=False)

### HD vs PC vs HDPC vs notHDnotPC

Plot Reactivation of cell assemblies

In [ ]:
fig, axs = plt.subplots(1, 5, figsize=(13,3))
axs = axs.flatten()
substates= ['SleepAfter', 'AW', 'QW', 'NREM', 'REM']
#df_CellAss = df_CellAss[df_CellAss['ExpeType']=='SleepAfter']

for plot, substate in enumerate(substates): 
    ax= axs[plot]

    df_CellAss_ID_ = df_CellAss_ID[(df_CellAss_ID['Substate'] == substate)] if not substate =='SleepAfter' else df_CellAss_ID
    CellAss_ID_pivot= df_CellAss_ID_.pivot_table(index='Assembly_ID', columns=['Assembly_Neurons_ID'], values='Avg_ReactStr', aggfunc='mean', fill_value=None)

    g = pd.DataFrame({"mean": np.nanmean(CellAss_ID_pivot, axis=0),"sem": CellAss_ID_pivot.sem(skipna=True),"count" : CellAss_ID_pivot.count()})
    g = g.reindex(['PC', 'PC&HD', 'HD', 'notPCnotHD', 'mix'])   
    color_map = {'PC': 'blue', 'PC&HD': 'orange', 'HD': 'green', 'notPCnotHD': 'black','mix': 'purple'} 
    colors = [color_map[c] for c in g.index]   # in order of shown categories
    for x,(m,e,c) in enumerate(zip(g['mean'], g['sem'], colors)):
        ax.errorbar(x, m, yerr=e, fmt='o', capsize=5, color=c)
    ax.set_ylabel('Reactivation Strength')
    ax.axhline(y=0, linestyle='--', color='grey')
    ax.set_xticks(range(len(g)), g.index, rotation=45, ha='right')
    ax.set_yticks((-.15,.15))
    ax.set_title(f"During {substate} \n (n = {g['count'].values.tolist()})")
    
    print(f'############################ {substate} ############################')    
    groups = [CellAss_ID_pivot[col].dropna().values for col in CellAss_ID_pivot.columns]
    f_stat, p_value = f_oneway(*groups)
    print(f"ANOVA result: F = {f_stat:.3f}, p = {p_value:.3e}")
    if p_value<0.05:
        df_melt = CellAss_ID_pivot.melt(var_name='Assembly_Neurons_ID', value_name='Avg_ReactStr')
        df_melt = df_melt.dropna(subset=['Avg_ReactStr', 'Assembly_Neurons_ID'])
        df_melt['Avg_ReactStr'] = pd.to_numeric(df_melt['Avg_ReactStr'], errors='coerce')
        tukey = pairwise_tukeyhsd(endog=df_melt['Avg_ReactStr'], groups=df_melt['Assembly_Neurons_ID'], alpha=0.05)
        print(tukey.summary())

plt.tight_layout()
plt.savefig(f'{Path(dfilepath).parent}/CellAssemblyRS_perVigSt&NeuronID.svg', format='svg', bbox_inches='tight', pad_inches=0, transparent=True)
plt.show()

Cell assembly size

In [ ]:
CellAss_ID_pivot= df_CellAss_ID.pivot_table(index='Assembly_ID', columns=['ExpeType'], values='Assembly_size', aggfunc='mean', fill_value=None)
CellAss_ID_pivot_clean = pd.merge(CellAss_ID_pivot,df_CellAss_ID[['Assembly_Neurons_ID', 'Assembly_ID']], on='Assembly_ID', how='inner')
CellAss_ID_pivot_clean = CellAss_ID_pivot_clean.drop_duplicates()

color_map = {'PC': 'blue', 'PC&HD': 'orange', 'HD': 'green', 
             'notPCnotHD': 'black', 'mix': 'purple'}

df_melted = CellAss_ID_pivot_clean.melt(
    id_vars='Assembly_Neurons_ID',
    value_vars=['SleepAfter'],
    var_name='ExpeType',
    value_name='Assembly_size'
)
df_melted['Assembly_Neurons_ID'] = pd.Categorical(df_melted['Assembly_Neurons_ID'], ["PC", "HD", "PC&HD", "notPCnotHD", "mix"])

fig, ax= plt.subplots(figsize=(15,2))
sns.swarmplot(
    x='ExpeType', 
    y='Assembly_size', 
    data=df_melted, 
    hue='Assembly_Neurons_ID', 
    palette=color_map, 
    size=6, 
    alpha=1,
    ax= ax,
)
plt.ylabel('Cell Assembly size')
plt.xticks([0], ['Cheeseboard'])
plt.legend(loc='upper right', title='Majority of')
plt.tight_layout()
plt.savefig(f'{Path(dfilepath).parent}/CellAssemblySize_perExpeType.svg', format='svg', bbox_inches='tight', pad_inches=0, transparent=True)
plt.show()


In [ ]:
fig, ax = plt.subplots(figsize=(2.5,2.5))
df_melted_ = df_melted.dropna().groupby(['ExpeType','Assembly_Neurons_ID']).size().reset_index(name='count')
pivot_df = df_melted_.pivot(index='ExpeType', columns='Assembly_Neurons_ID', values='count').fillna(0)
size = 0.5
vals_outer = df_melted_[df_melted_['ExpeType']=='SleepAfter']['count'].tolist()
color_map = ['blue', 'green', 'orange', 'black', 'purple']

ax.pie(vals_outer, radius=1.2, colors=color_map, #labels=['Spatial', 'Non_Spatial', 'unclassified', 'mix'],
       wedgeprops=dict(width=size, edgecolor='w'))
ax.set(aspect="equal", title= 'Cheeseboard')
plt.savefig(f'{Path(dfilepath).parent}/CellAssemblyProportion_NestedPieChart_perExpeType.svg', format='svg', bbox_inches='tight', pad_inches=0, transparent=True)
plt.show()

### Spatially-tuned vs non Spatially-tuned

Plot Reactivation of cell assemblies

In [ ]:
fig, axs = plt.subplots(1, 5, figsize=(10,3))
axs = axs.flatten()
substates= ['SleepAfter', 'AW', 'QW', 'NREM', 'REM']
for plot, substate in enumerate(substates): 
    ax= axs[plot]

    df_CellAss_ID_ = df_CellAss_ID[(df_CellAss_ID['Substate'] == substate)] if not substate =='SleepAfter' else df_CellAss_ID
    CellAss_ID_pivot= df_CellAss_ID_.pivot_table(index='Assembly_ID', columns=['Assembly_Spatial_ID'], values='Avg_ReactStr', aggfunc='mean', fill_value=None)

    g = pd.DataFrame({"mean": np.nanmean(CellAss_ID_pivot, axis=0),"sem": CellAss_ID_pivot.sem(skipna=True),"count" : CellAss_ID_pivot.count()})
    g = g.reindex(['Spatial', 'Non_Spatial','mix'])   
    color_map = {'Spatial': 'blue', 'Non_Spatial': 'orange', 'mix': 'purple'} 
    colors = [color_map[c] for c in g.index]   # in order of shown categories
    for x,(m,e,c) in enumerate(zip(g['mean'], g['sem'], colors)):
        ax.errorbar(x, m, yerr=e, fmt='o', capsize=5, color=c)
    ax.set_ylabel('Reactivation Strength')
    ax.axhline(y=0, linestyle='--', color='grey')
    ax.set_xticks(range(len(g)), g.index, rotation=45, ha='right')
    ax.set_yticks((-.15,.15))
    ax.set_title(f"During {substate} \n (n = {g['count'].values.tolist()})")
    
    print(f'############################ {substate} ############################')    
    groups = [CellAss_ID_pivot[col].dropna().values for col in CellAss_ID_pivot.columns]
    f_stat, p_value = f_oneway(*groups)
    print(f"ANOVA result: F = {f_stat:.3f}, p = {p_value:.3e}")
    if p_value<0.05: 
        df_melt = CellAss_ID_pivot.melt(var_name='Assembly_Spatial_ID', value_name='Avg_ReactStr')
        df_melt = df_melt.dropna(subset=['Avg_ReactStr', 'Assembly_Spatial_ID'])
        df_melt['Avg_ReactStr'] = pd.to_numeric(df_melt['Avg_ReactStr'], errors='coerce')
        tukey = pairwise_tukeyhsd(endog=df_melt['Avg_ReactStr'], groups=df_melt['Assembly_Spatial_ID'], alpha=0.05)
        print(tukey.summary())

plt.tight_layout()
plt.savefig(f'{Path(dfilepath).parent}/CellAssemblyRS_perVigSt&SpatialID.svg', format='svg', bbox_inches='tight', pad_inches=0, transparent=True)
plt.show()

Cell assembly size

In [ ]:
CellAss_ID_pivot = df_CellAss_ID.pivot_table(index='Assembly_ID', columns=['ExpeType'], values='Assembly_size', aggfunc='mean', fill_value=None)
CellAss_ID_pivot_clean = pd.merge(CellAss_ID_pivot,df_CellAss_ID[['Assembly_Spatial_ID', 'Assembly_ID']], on='Assembly_ID', how='inner')
CellAss_ID_pivot_clean = CellAss_ID_pivot_clean.drop_duplicates()

color_map = {'Spatial': 'blue', 'Non_Spatial': 'orange', 'mix': 'purple'} 

df_melted = CellAss_ID_pivot_clean.melt(
    id_vars='Assembly_Spatial_ID',
    value_vars=['SleepAfter'],
    var_name='ExpeType',
    value_name='Assembly_size'
)
df_melted['Assembly_Spatial_ID'] = pd.Categorical(df_melted['Assembly_Spatial_ID'], ["Spatial", "Non_Spatial", "mix"])

fig, ax= plt.subplots(figsize=(15,2))
sns.swarmplot(
    x='ExpeType', 
    y='Assembly_size', 
    data=df_melted, 
    hue='Assembly_Spatial_ID', 
    palette=color_map, 
    size=6, 
    alpha=1,
    ax= ax,
)
plt.ylabel('Cell Assembly size')
plt.xticks([0], ['Cheeseboard'])
plt.legend(loc='upper right', title='Majority of')
plt.tight_layout()
plt.savefig(f'{Path(dfilepath).parent}/CellAssemblySize_Spatial_perExpeType.svg', format='svg', bbox_inches='tight', pad_inches=0, transparent=True)
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(2.5,2.5))
df_melted_ = df_melted.dropna().groupby(['ExpeType','Assembly_Spatial_ID']).size().reset_index(name='count')
pivot_df = df_melted_.pivot(index='ExpeType', columns='Assembly_Spatial_ID', values='count').fillna(0)
size = .5
vals_outer = df_melted_[df_melted_['ExpeType']=='SleepAfter']['count'].tolist()
color_map = ['blue', 'orange', 'purple',]

ax.pie(vals_outer, radius=1.2, colors=color_map, #labels=['Spatial', 'Non_Spatial', 'unclassified', 'mix'],
       wedgeprops=dict(width=size, edgecolor='w'))
ax.set(aspect="equal", title= 'Cheeseboard')
plt.savefig(f'{Path(dfilepath).parent}/CellAssemblyProportion_Spatial_NestedPieChart_perExpeType.svg', format='svg', bbox_inches='tight', pad_inches=0, transparent=True)
plt.show()

# Reactivation around oscillations

In [ ]:
dfilepath=f"//10.69.168.1/crnldata/forgetting/Aurelie/MiniscopeOE_analysis/PlaceCells_experiment/{Analysisfolder}/CellAssembly_Global_ID.pkl"
with open(dfilepath, 'rb') as pickle_file:
    df_CellAss_ID = pickle.load(pickle_file)

In [ ]:
anaylisfolder= '4_OscReactivation_2025-11-07_16_21_11_calcium_nozscore'
path=f"//10.69.168.1/crnldata/forgetting/Aurelie/MiniscopeOE_analysis/PlaceCells_experiment/{anaylisfolder}/"


Save Global in .csv for R

In [ ]:
with open(f"{path}/SWR_Global.pkl", "rb") as f:
    obj = pickle.load(f)
obj = pd.merge(obj, df_CellAss_ID[['Assembly_ID', 'Assembly_Spatial_ID', 'Assembly_Neurons_ID']].drop_duplicates(), on='Assembly_ID', how='outer')
obj.to_csv(f"{path}/SWR_Global.csv", index=False)


with open(f"{path}/Spdl_Global.pkl", "rb") as f:
    obj = pickle.load(f)
obj = pd.merge(obj, df_CellAss_ID[['Assembly_ID', 'Assembly_Spatial_ID', 'Assembly_Neurons_ID']].drop_duplicates(), on='Assembly_ID', how='outer')
obj.to_csv(f"{path}/Spdl_Global.csv", index=False)

Select Coupling

In [ ]:
Oscillations={'SPDL', 'SWR'}

Data='Ca'
Drug='baseline'
Coupling='Precoupled'
save = 0

for Osc in Oscillations:
    if Osc == 'SWR':
        Ctx= {'CA1', 'RSC','CA1RSC'}
    elif Osc == 'SPDL': 
        Ctx= {'S1', 'PFC','S1PFC'}
    for ctx in Ctx:
        filename=f"{path}/{Osc}_{Data}PSTH_{Coupling}{ctx}baseline.pkl"
        with open(filename, 'rb') as pickle_file:
            locals()[f'{Osc}_{ctx}'] = pickle.load(pickle_file)

OR gather Pre, Post, PrePostcoupled oscillations into Coupled

In [ ]:
Couplings={'Precoupled', 'PrePostcoupled', 'Postcoupled'}
Coupling= 'Coupled'

Oscillations={'SPDL', 'SWR'}

Data='Ca'
Drug='baseline'

for Osc in Oscillations:
    if Osc == 'SWR':
        Ctx= {'CA1', 'RSC','CA1RSC'}
    elif Osc == 'SPDL': 
        Ctx= {'S1', 'PFC','S1PFC'}
    for ctx in Ctx:        
        for coup in Couplings:
            filename=f"{path}/{Osc}_{Data}PSTH_{coup}{ctx}baseline.pkl"
            with open(filename, 'rb') as pickle_file:
                locals()[f'{Osc}_{ctx}_{coup}'] = pickle.load(pickle_file)
        dicts= [locals()[f'{Osc}_{ctx}_Precoupled'], locals()[f'{Osc}_{ctx}_PrePostcoupled'], locals()[f'{Osc}_{ctx}_Postcoupled']]
        locals()[f'{Osc}_{ctx}'] = defaultdict(list)
        for d in dicts:
            for key, value in d.items():
                locals()[f'{Osc}_{ctx}'][key].extend(value)  # extend because values are lists of lists
        locals()[f'{Osc}_{ctx}'] = dict(locals()[f'{Osc}_{ctx}'])

Example from one cell assembly

In [ ]:
cell_ass='13_00_01_1_CellAss0'
save = 1

In [ ]:
def process_and_plot(data, ax, vmin=-5, vmax=5):
    #data1=data.iloc[:, data.shape[1] // 10 *4  : data.shape[1] // 10 * 6]
    data1=data[:, data.shape[1] // 10 *4  : data.shape[1] // 10 * 6]
    n_cols = np.shape(data1)[1] // 4  # Floor division to get the number of columns in 1/4th
    baseline_columns = data1[:, :n_cols]  # Select the first 'n_cols' columns
    mean_baseline = np.nanmean(baseline_columns,axis=1)
    data2 = data1-mean_baseline[:, np.newaxis]
    sns.heatmap(data, ax=ax, cmap='viridis', xticklabels=False, yticklabels=False, vmin=vmin, vmax=vmax)
    #sns.heatmap(data2, ax=ax, cmap='viridis', xticklabels=False, yticklabels=False, vmin=vmin, vmax=vmax)

plt.close()

for Osc in Oscillations:
    if Osc == 'SWR':
        Ctx= {'CA1', 'RSC','CA1RSC'}
    elif Osc == 'SPDL': 
        Ctx= {'S1', 'PFC','S1PFC'}
    fig, axes = plt.subplots(1, len(Ctx), figsize=(6, 3))
    # Check if subplots are correctly created
    for subplotnb, ctx in enumerate(Ctx):
        PSTH = locals()[f'{Osc}_{ctx}']
        axes[subplotnb].set_title(f'{Osc}_{ctx}', size=10)
        if PSTH != {}: 
            process_and_plot(PSTH[cell_ass], axes[subplotnb])
    fig.suptitle(f"{cell_ass} ({df_CellAss_ID[df_CellAss_ID['Assembly_ID']==cell_ass]['Assembly_Neurons_ID'].unique().tolist()[0]}, {df_CellAss_ID[df_CellAss_ID['Assembly_ID']==cell_ass]['Assembly_Spatial_ID'].unique().tolist()[0]})")    
    plt.tight_layout()
    plt.savefig(f'{Path(path)}/{cell_ass}_heatmap_{Osc}.svg', format='svg', bbox_inches='tight', pad_inches=0, transparent=True) if save else None
    plt.show()


In [ ]:
plt.close()

cmap = sns.light_palette("#09008B", as_cmap=True)
#cmap = sns.light_palette("black", as_cmap=True)

def plot_lines(data, ax, title, xlabel=None, title_color='black', cmap=cmap):
    num_lines = np.shape(data)[0]
    colors = cmap(np.linspace(0., .8, num_lines))
    ax.text(-1,0, f'#1', color='black', ha='right', va='center', fontsize=8) 
    for i in range(num_lines):
        line_data = (data[i])/max(data[i]) - i * 1.5 if sum(data[i]) != 0 else data[i] - i * 1.5
        ax.plot(0, min(line_data), line_data, color=colors[i], linewidth=1)
    ax.text(-1, min(line_data), f'#{i+1}', color='black', ha='right', va='center', fontsize=8) 
    ax.set_title(title, fontsize=10, color=title_color, pad=10)  # Added padding for the title
    if xlabel:
        ax.set_xlabel(xlabel, labelpad=2, fontsize=10)    
    ax.axis('off')  # Hide axis lines and labels

# Check if subplots are correctly created

colortitle= ['orange','rebeccapurple','mediumseagreen',] #if Osc =='SPDL' else ['orange','rebeccapurple','mediumseagreen',]
for Osc in Oscillations:
    if Osc == 'SWR':
        Ctx= {'CA1', 'RSC','CA1RSC'}
    elif Osc == 'SPDL': 
        Ctx= {'S1', 'PFC','S1PFC'}
    fig, axes = plt.subplots(1,  len(Ctx), figsize=(6, 3))  # Increased figure size for better readability
    for subplotnb, ctx in enumerate(Ctx):
        PSTH = locals()[f'{Osc}_{ctx}']
        data = PSTH[cell_ass][::]*1   #[::2] Taking every other row
        plot_lines(data, axes[subplotnb], f'{ctx}', title_color=colortitle[subplotnb])

    fig.suptitle(f"{cell_ass} ({df_CellAss_ID[df_CellAss_ID['Assembly_ID']==cell_ass]['Assembly_Neurons_ID'].unique().tolist()[0]}, {df_CellAss_ID[df_CellAss_ID['Assembly_ID']==cell_ass]['Assembly_Spatial_ID'].unique().tolist()[0]})")    
    plt.tight_layout()
    plt.savefig(f'{Path(path)}/{cell_ass}_RSTraces_{Osc}.svg', format='svg', bbox_inches='tight', pad_inches=0, transparent=True) if save else None
    plt.show()


In [ ]:
plt.close()

#cmap = sns.light_palette("#008B8B", as_cmap=True)
cmap = sns.light_palette("black", as_cmap=True)

def plot_lines(data, ax, title, title_color='black', cmap=cmap):
    data1=data[:, data.shape[1] // 10 *4  : data.shape[1] // 10 * 6]
    n_cols = np.shape(data1)[1]// 4  # Floor division to get the number of columns in 1/4th
    baseline_columns = data1[:, :n_cols]   # Select the first 'n_cols' columns
    mean_baseline = np.nanmean(baseline_columns,axis=1)
    data2 = data1-mean_baseline[:, np.newaxis]
    ax.plot(np.nanmean(data2, axis=0), color=title_color, linewidth=2)
    ax.set_ylim(-1, 2)    
    ax.set_title(title, fontsize=10, color=title_color, pad=10)  # Added padding for the title
    ax.axis('off')  # Hide axis lines and labels

for Osc in Oscillations:
    if Osc == 'SWR':
        Ctx= {'CA1', 'RSC','CA1RSC'}
    elif Osc == 'SPDL': 
        Ctx= {'S1', 'PFC','S1PFC'}
    fig, axes = plt.subplots(1, len(Ctx), figsize=(6, 2))  # Increased figure size for better readability

    colortitle= ['orange','rebeccapurple','mediumseagreen',] #if Osc =='SPDL' else ['orange','rebeccapurple','mediumseagreen',]
    for subplotnb, ctx in enumerate(Ctx):
        PSTH = locals()[f'{Osc}_{ctx}']
        data = PSTH[cell_ass][::]*1   #[::2] Taking every other row
        plot_lines(data, axes[subplotnb], f'{ctx}', title_color=colortitle[subplotnb])

    fig.suptitle(f"{cell_ass} ({df_CellAss_ID[df_CellAss_ID['Assembly_ID']==cell_ass]['Assembly_Neurons_ID'].unique().tolist()[0]}, {df_CellAss_ID[df_CellAss_ID['Assembly_ID']==cell_ass]['Assembly_Spatial_ID'].unique().tolist()[0]})")    

    plt.tight_layout()
    plt.savefig(f'{Path(path)}/{cell_ass}_AvgRSTraces_{Osc}.svg', format='svg', bbox_inches='tight', pad_inches=0, transparent=True) if save else None
    plt.show()


Mean for each cell assemblies

In [ ]:
def process_and_plot(data1, ax, vmin=-.5, vmax=.5):
    #data1=data.iloc[:, data.shape[1] // 10 *4  : data.shape[1] // 10 * 6]
    n_cols = np.shape(data1)[1] // 4  # Floor division to get the number of columns in 1/4th
    baseline_columns = data1[:, :n_cols]  # Select the first 'n_cols' columns
    mean_baseline = np.nanmean(baseline_columns,axis=1)
    data2 = data1-mean_baseline[:, np.newaxis]
    sns.heatmap(data2, ax=ax, cmap='viridis', xticklabels=False, yticklabels=False, vmin=vmin, vmax=vmax)
    #sns.heatmap(data2, ax=ax, cmap='viridis', xticklabels=False, yticklabels=False, vmin=vmin, vmax=vmax)

plt.close()

for Osc in Oscillations:
    if Osc == 'SWR':
        Ctx= {'CA1', 'RSC','CA1RSC'}
    elif Osc == 'SPDL': 
        Ctx= {'S1', 'PFC','S1PFC'}
    fig, axes = plt.subplots(1, len(Ctx), figsize=(6, 3))

    # Check if subplots are correctly created
    for subplotnb, ctx in enumerate(Ctx):
        PSTH = locals()[f'{Osc}_{ctx}']
        PSTH_filt = {k: v for k, v in PSTH.items() if k in df_CellAss_ID['Assembly_ID'].values}
        axes[subplotnb].set_title(f'{Osc}_{ctx}', size=10)
        if PSTH_filt != {}: 
            data=np.array([np.nanmean(PSTH_filt[key], axis= 0) for key in PSTH_filt.keys()])
            process_and_plot(data, axes[subplotnb])
    fig.suptitle(f"All cell assemblies (n = {len(PSTH_filt.keys())})")    
    plt.tight_layout()
    plt.savefig(f'{Path(path)}/AllCellAss_heatmap_{Coupling}{Osc}.svg', format='svg', bbox_inches='tight', pad_inches=0, transparent=True) if save else None
    plt.show()


In [ ]:
plt.close()

#cmap = sns.light_palette("#008B8B", as_cmap=True)
cmap = sns.light_palette("black", as_cmap=True)

def plot_lines(data1, ax, title, title_color='black', cmap=cmap):
    n_cols = np.shape(data1)[1]// 4  # Floor division to get the number of columns in 1/4th
    baseline_columns = data1[:, :n_cols]   # Select the first 'n_cols' columns
    mean_baseline = np.nanmean(baseline_columns,axis=1)
    data2 = data1-mean_baseline[:, np.newaxis]
    ax.plot(np.nanmean(data2, axis=0), color=title_color, linewidth=2)
    ax.set_ylim(-5, 5)    
    ax.set_title(title, fontsize=10, color=title_color, pad=10)  # Added padding for the title
    ax.axis('off')  # Hide axis lines and labels

for Osc in Oscillations:
    if Osc == 'SWR':
        Ctx= {'CA1', 'RSC','CA1RSC'}
    elif Osc == 'SPDL': 
        Ctx= {'S1', 'PFC','S1PFC'}
    fig, axes = plt.subplots(1, len(Ctx), figsize=(6, 2))  # Increased figure size for better readability

    colortitle= ['orange','rebeccapurple','mediumseagreen',] #if Osc =='SPDL' else ['orange','rebeccapurple','mediumseagreen',]
    for subplotnb, ctx in enumerate(Ctx):
        PSTH = locals()[f'{Osc}_{ctx}']
        PSTH_filt = {k: v for k, v in PSTH.items() if k in df_CellAss_ID['Assembly_ID'].values}
        if PSTH_filt != {}: 
            data=np.array([np.nanmean(PSTH_filt[key], axis= 0) for key in PSTH_filt.keys()])
            plot_lines(data*10, axes[subplotnb], f'{ctx}', title_color=colortitle[subplotnb])

    fig.suptitle(f"All cell assemblies (n = {len(PSTH_filt.keys())})")    

    plt.tight_layout()
    plt.savefig(f'{Path(path)}/AllCellAss_AvgRStraces_{Coupling}{Osc}.svg', format='svg', bbox_inches='tight', pad_inches=0, transparent=True) if save else None
    plt.show()


Per Neuron ID

In [ ]:
label_colors = {'PC': 'blue', 'PC&HD': 'orange', 'HD': 'green', 'notPCnotHD': 'grey', 'mix': 'purple'}
label_colors2 = {'BC': 'cyan', 'RC': 'red', 'RL': 'magenta', 'YL': 'yellow'}

window = [-1, 1]  # from -10s to +10s around each event
bin_width = 1/20
bins = np.arange(window[0], window[1] + bin_width, bin_width)

for Osc in Oscillations:
    if Osc == 'SWR':
        Ctx= {'CA1', 'RSC','CA1RSC'}
    elif Osc == 'SPDL': 
        Ctx= {'S1', 'PFC','S1PFC'}
        
    fig = plt.figure(figsize=(10, 4))    
    subfigs = fig.subfigures(1, len(Ctx), wspace=0.07, hspace=0.1)

    print(Osc)

    for subplotnb, subfig in enumerate(subfigs.flat, start=1):
        ctx=list(Ctx)[subplotnb-1]  
        
        PSTH = locals()[f'{Osc}_{ctx}']
        PSTH_filt = {k: v for k, v in PSTH.items() if k in df_CellAss_ID['Assembly_ID'].values} #keep Cell assemblies with >=2 cells in it
        PSTH_filt_norm = {key: data - np.nanmean(data[:, :data.shape[1] // 4], axis=1)[:, None] for key, data in PSTH_filt.items()}
        data = np.array([np.nanmean(PSTH_filt_norm[key], axis= 0) for key in PSTH_filt_norm.keys()])
        psth_df = pd.DataFrame(data.T, columns = list(PSTH_filt_norm.keys()))
        psth_df_n = psth_df #/ psth_df.max() 
        psth_df_n = psth_df_n.replace(np.nan, 0)
        max_row_pos = psth_df_n.values.argmax(axis=0)
        col_sums = psth_df_n.sum()
        col_info = pd.DataFrame({'col': psth_df_n.columns,'max_pos': max_row_pos,'sum': col_sums.values})
        col_info_sorted = col_info.sort_values(by=['max_pos', 'sum'])
        psth_df_s = psth_df_n[col_info_sorted['col'].values]
        psth_df_s = psth_df_s.loc[:, (psth_df_s != 0).any(axis=0)]
        df = psth_df_s.T

        df['Assembly_ID']=df.index
        df = pd.merge(df, df_CellAss_ID[['Assembly_ID', 'Mice', 'Assembly_Neurons_ID']].drop_duplicates(), on='Assembly_ID', how='outer')
        df["Assembly_Neurons_ID"] = df["Assembly_Neurons_ID"].str.replace("_", "", regex=False)
        df.index=  df["Mice"] + "_" + df["Assembly_ID"] + "_" + df["Assembly_Neurons_ID"]
        df = df.drop(["Assembly_ID", "Assembly_Neurons_ID", "Mice"], axis=1)

        gs = subfig.add_gridspec(nrows=2, ncols=3, height_ratios=[3, 1], width_ratios=[10, 0.5, 0.5],hspace=0.01, wspace=0.05)

        row_labels = df.index.str.split('_').str[-1]
        color_codes = [list(label_colors).index(lbl) for lbl in row_labels]
        color_array = np.array(color_codes).reshape(-1, 1)
        cmap = ListedColormap([label_colors[k] for k in label_colors])
        
        row_labels2 = df.index.str.split('_').str[0]
        color_codes2 = [list(label_colors2).index(lbl) for lbl in row_labels2]
        color_array2 = np.array(color_codes2).reshape(-1, 1)
        cmap2 = ListedColormap([label_colors2[k] for k in label_colors2])

        ax_heatmap = subfig.add_subplot(gs[0, 0])
        sns.heatmap(df, ax=ax_heatmap, cmap="viridis", cbar=False, cbar_kws={'location': 'top'}, yticklabels=False, vmin=0, vmax=.2)
        ax_heatmap.set_yticklabels([]) 
        ax_heatmap.set_xticks([])
        ax_heatmap.set_ylabel(f" Cell assemblies")  
        ax_heatmap.collections[0].set_rasterized(True) if save else None

        ax_ann = subfig.add_subplot(gs[0, 1])
        sns.heatmap(color_array, ax=ax_ann, cmap=cmap, cbar=False)
        ax_ann.set_xticks([])
        ax_ann.set_yticks([])
        ax_ann.tick_params(left=False, right=False)       

        ax_ann2 = subfig.add_subplot(gs[0, 2])
        sns.heatmap(color_array2, ax=ax_ann2, cmap='Set3', cbar=False)
        ax_ann2.set_xticks([])
        ax_ann2.set_yticks([])
        ax_ann2.tick_params(left=False, right=False)             

        ax_bottom = subfig.add_subplot(gs[1, 0])  
        #ax_bottom.plot(time_bins, zscore(np.mean(df, axis=0)), color='black')
        time_bins = np.linspace(window[0], window[1], len(bins)-1)
        ax_bottom.axvline(x=0, color='lightgrey', linestyle='--', linewidth=1)
        for label in label_colors.keys():
            df_a = df[df.index.str.split('_').str[-1] == label]
            ax_bottom.plot(time_bins, zscore(np.mean(df_a, axis=0)), color=label_colors[label])
        ax_bottom.set_xlabel(f"Time from {Osc} (s)")
        ax_bottom.set_xlim(window[0], window[1])
        ax_bottom.set_ylim(-3, 3)

        subfig.suptitle(f'{Coupling} {ctx} {Osc} (n = {len(df)})')
        
    plt.tight_layout()
    plt.savefig(f'{Path(path)}/AllCellAss_RSheatmap&traces_{Coupling}{Osc}.svg', format='svg', bbox_inches='tight', pad_inches=0, transparent=True) if save else None
    plt.show()

Per Spatial ID

In [ ]:
label_colors = {'Spatial': 'blue', 'NonSpatial': 'orange', 'mix': 'purple'}
window = [-1, 1]  # from -10s to +10s around each event
bin_width = 1/20
bins = np.arange(window[0], window[1] + bin_width, bin_width)

for Osc in Oscillations:
    if Osc == 'SWR':
        Ctx= {'CA1', 'RSC','CA1RSC'}
    elif Osc == 'SPDL': 
        Ctx= {'S1', 'PFC','S1PFC'}
        
    fig = plt.figure(figsize=(10, 4))    
    subfigs = fig.subfigures(1, len(Ctx), wspace=0.07, hspace=0)
    print(Osc)

    for subplotnb, subfig in enumerate(subfigs.flat, start=1):
        ctx=list(Ctx)[subplotnb-1]     
        
        PSTH = locals()[f'{Osc}_{ctx}']
        PSTH_filt = {k: v for k, v in PSTH.items() if k in df_CellAss_ID['Assembly_ID'].values}
        PSTH_filt_norm = {key: data - np.nanmean(data[:, :data.shape[1] // 4], axis=1)[:, None] for key, data in PSTH_filt.items()}
        data = np.array([np.nanmean(PSTH_filt_norm[key], axis= 0) for key in PSTH_filt_norm.keys()])
        psth_df = pd.DataFrame(data.T, columns = list(PSTH_filt_norm.keys()))
        psth_df_n = psth_df# / psth_df.max() 
        psth_df_n = psth_df_n.replace(np.nan, 0)
        max_row_pos = psth_df_n.values.argmax(axis=0)
        col_sums = psth_df_n.sum()
        col_info = pd.DataFrame({'col': psth_df_n.columns,'max_pos': max_row_pos,'sum': col_sums.values})
        col_info_sorted = col_info.sort_values(by=['max_pos', 'sum'])
        psth_df_s = psth_df_n[col_info_sorted['col'].values]
        psth_df_s = psth_df_s.loc[:, (psth_df_s != 0).any(axis=0)]
        df = psth_df_s.T

        df['Assembly_ID']=df.index
        df = pd.merge(df, df_CellAss_ID[['Assembly_ID', 'Mice', 'Assembly_Spatial_ID']].drop_duplicates(), on='Assembly_ID', how='outer')
        df["Assembly_Spatial_ID"] = df["Assembly_Spatial_ID"].str.replace("_", "", regex=False)

        df.index=  df["Mice"] + "_" + df["Assembly_ID"] + "_" + df["Assembly_Spatial_ID"]
        df = df.drop(["Assembly_ID", "Assembly_Spatial_ID", "Mice"], axis=1)

        if len(df)>0:

            gs = subfig.add_gridspec(nrows=2, ncols=3, height_ratios=[3, 1],   width_ratios=[10, 0.5, 0.5], hspace=0.01, wspace=0.05 )

            row_labels = df.index.str.split('_').str[-1]
            color_codes = [list(label_colors).index(lbl) for lbl in row_labels]
            color_array = np.array(color_codes).reshape(-1, 1)
            cmap1 = ListedColormap([label_colors[k] for k in label_colors])
            
            row_labels2 = df.index.str.split('_').str[0]
            color_codes2 = [list(label_colors2).index(lbl) for lbl in row_labels2]
            color_array2 = np.array(color_codes2).reshape(-1, 1)
            cmap2 = ListedColormap([label_colors2[k] for k in label_colors2])

            ax_heatmap = subfig.add_subplot(gs[0, 0])
            sns.heatmap(df, ax=ax_heatmap, cmap="viridis", cbar=False, cbar_kws={'location': 'top'}, yticklabels=False, vmin=0, vmax=.2)
            ax_heatmap.set_yticklabels([]) 
            ax_heatmap.set_xticks([])
            ax_heatmap.set_ylabel(f" Cell assemblies")  
            ax_heatmap.collections[0].set_rasterized(True) if save else None
            
            ax_ann = subfig.add_subplot(gs[0, 1])
            sns.heatmap(color_array, ax=ax_ann, cmap=cmap1, cbar=False)
            ax_ann.set_xticks([])
            ax_ann.set_yticks([])
            ax_ann.tick_params(left=False, right=False)       

            ax_ann2 = subfig.add_subplot(gs[0, 2])
            sns.heatmap(color_array2, ax=ax_ann2, cmap='Set3', cbar=False)
            ax_ann2.set_xticks([])
            ax_ann2.set_yticks([])
            ax_ann2.tick_params(left=False, right=False) 

            time_bins = np.linspace(window[0], window[1], len(bins)-1)
            ax_bottom = subfig.add_subplot(gs[1, 0])  # span both columns
            #ax_bottom.plot(time_bins, zscore(np.mean(df, axis=0)), color='black')
            ax_bottom.axvline(x=0, color='lightgrey', linestyle='--', linewidth=1)
            for label in label_colors.keys():
                df_a = df[df.index.str.split('_').str[-1] == label]
                ax_bottom.plot(time_bins, (np.mean(df_a, axis=0)), color=label_colors[label])
            ax_bottom.set_xlabel(f"Time from {ctx} {Osc} (s)")
            ax_bottom.set_xlim(window[0], window[1])
            #ax_bottom.set_ylim(-3, 3)

            subfig.suptitle(f'{Coupling} {ctx} {Osc} (n = {len(df)})')
    plt.tight_layout()
    plt.savefig(f'{Path(path)}/AllCellAss_Spatial_RSheatmap&traces_{Coupling}{Osc}.svg', format='svg', bbox_inches='tight', pad_inches=0, transparent=True) if save else None
    plt.show()